In [ ]:
# Salwa
import pandas as pd

RAW_PATH = "../data/raw_crashes.csv"
OUT_PATH = "../data/cleaned_crashes.csv.gz"

df = pd.read_csv(RAW_PATH, low_memory=False)
df.columns = df.columns.str.lower()
rows_raw = len(df)

first_date = str(df["crash_date"].iloc[0])
if "T" in first_date:
    date_format = "ISO8601"
else:
    date_format = "%m/%d/%Y %I:%M:%S %p"
df["crash_date"] = pd.to_datetime(df["crash_date"], format=date_format)
df["crash_year"] = df["crash_date"].dt.year

df = df.drop_duplicates(subset="crash_record_id")
df = df.replace("UNKNOWN", pd.NA)

bad_loc = (df["latitude"] == 0) | (df["longitude"] == 0)
df.loc[bad_loc, ["latitude", "longitude"]] = pd.NA

df["posted_speed_limit"] = pd.to_numeric(df["posted_speed_limit"], errors="coerce")
df.loc[~df["posted_speed_limit"].between(5, 70), "posted_speed_limit"] = pd.NA

df = df.dropna(subset=["most_severe_injury"])

for c in ["injuries_fatal", "injuries_incapacitating", "injuries_total"]:
    df[c] = df[c].fillna(0)

df["severe"] = ((df["injuries_fatal"] + df["injuries_incapacitating"]) > 0).astype(int)
df["any_injury"] = (df["injuries_total"] > 0).astype(int)
df["hit_and_run"] = (df["hit_and_run_i"] == "Y").astype(int)

rows_before_year_filter = len(df)
df = df[df["crash_year"] >= 2018].reset_index(drop=True)

print("Raw rows:", rows_raw)
print("After removing duplicates and rows with no injury info:", rows_before_year_filter)
print("After keeping 2018 onward:", len(df))
print("Share of severe crashes:", round(df["severe"].mean(), 4))
print(df["crash_year"].value_counts().sort_index())
print(df[["lighting_condition", "weather_condition", "posted_speed_limit", "latitude"]].isna().sum())

df.to_csv(OUT_PATH, index=False)
print("Saved to", OUT_PATH)

Raw rows: 1108694
After removing duplicates and rows with no injury info: 1106297
After keeping 2018 onward: 968596
Share of severe crashes: 0.0179
crash_year
2018    118728
2019    117499
2020     91860
2021    108519
2022    108119
2023    110493
2024    111805
2025    108902
2026     92671
Name: count, dtype: int64
lighting_condition    50653
weather_condition     62552
posted_speed_limit     4673
latitude               8190
dtype: int64
